# Compare Infrared SDK vegetation and AI-detected trees

This notebook compares vegetation fetched from Infrared/OSM with Roboflow AI-detected tree points for the current Vienna orthophoto area. It merges both sources and removes near-duplicates so the same tree is not counted twice.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from backend.app.imagery.vienna_orthofoto import create_bbox_polygon
from backend.app.simulation.infrared_runner import (
    create_infrared_client,
    fetch_sdk_context,
    get_infrared_debug_info,
    merge_detected_and_sdk_vegetation,
)

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
detected_path = OUTPUTS_DIR / 'detected_trees.geojson'
metadata_path = DATA_DIR / 'vienna_orthofoto_test_metadata.json'

debug = get_infrared_debug_info()
debug

In [ ]:
if not detected_path.exists():
    raise FileNotFoundError(f'Missing {detected_path}. Run tree detection first.')
if not metadata_path.exists():
    raise FileNotFoundError(f'Missing {metadata_path}. Run notebook 01b first.')

detected_geojson = json.loads(detected_path.read_text(encoding='utf-8'))
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
polygon = create_bbox_polygon(metadata['bbox_lonlat'])
print('AI-detected tree count:', len(detected_geojson.get('features', [])))

In [ ]:
if not debug['infrared_available']:
    raise RuntimeError(debug['message'])

with create_infrared_client() as client:
    context = fetch_sdk_context(client, polygon)

merged_vegetation, merge_summary = merge_detected_and_sdk_vegetation(
    sdk_vegetation=context['sdk_vegetation'],
    detected_tree_geojson=detected_geojson,
)
merge_summary['ground_material_count'] = context['ground_material_count']
merge_summary

In [ ]:
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
merged_path = OUTPUTS_DIR / 'merged_vegetation.geojson'
summary_path = OUTPUTS_DIR / 'vegetation_merge_summary.json'

merged_fc = {'type': 'FeatureCollection', 'features': list(merged_vegetation.values())}
merged_path.write_text(json.dumps(merged_fc, indent=2), encoding='utf-8')
summary_path.write_text(json.dumps(merge_summary, indent=2), encoding='utf-8')

print('SDK/OSM tree count:', merge_summary['sdk_tree_count'])
print('AI-detected tree count:', merge_summary['detected_tree_count'])
print('Duplicate count:', merge_summary['duplicate_count'])
print('Merged tree count:', merge_summary['merged_tree_count'])
print('Saved:', merged_path)
print('Saved:', summary_path)

In [ ]:
import folium

center = metadata['center']
m = folium.Map(location=[center['lat'], center['lon']], zoom_start=18, tiles='OpenStreetMap')

sdk_group = folium.FeatureGroup(name='SDK / OSM vegetation', show=True)
for feature in context['sdk_vegetation'].values() if isinstance(context['sdk_vegetation'], dict) else context['sdk_vegetation']:
    geom = feature.get('geometry', {})
    if geom.get('type') != 'Point':
        continue
    lon, lat = geom['coordinates'][:2]
    folium.CircleMarker([lat, lon], radius=3, color='blue', fill=True, fill_opacity=0.8).add_to(sdk_group)
sdk_group.add_to(m)

detected_group = folium.FeatureGroup(name='AI-detected trees', show=True)
for feature in detected_geojson.get('features', []):
    geom = feature.get('geometry', {})
    if geom.get('type') != 'Point':
        continue
    lon, lat = geom['coordinates'][:2]
    folium.CircleMarker([lat, lon], radius=4, color='limegreen', fill=True, fill_opacity=0.9).add_to(detected_group)
detected_group.add_to(m)

merged_group = folium.FeatureGroup(name='Merged vegetation', show=False)
folium.GeoJson(merged_fc, name='Merged vegetation').add_to(merged_group)
merged_group.add_to(m)

folium.LayerControl().add_to(m)
map_path = OUTPUTS_DIR / 'vegetation_comparison_map.html'
m.save(map_path)
print('Saved:', map_path)
m